In [1]:
import os
import re
from datetime import datetime

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm import tqdm

In [2]:
# Configuration
START_DATE = "2003-01-02"
END_DATE = "2026-03-15"

print(f"Downloading daily data from {START_DATE} to {END_DATE}")

In [3]:
# Setup output directory
BASE = os.path.abspath(os.getcwd())
if os.path.basename(BASE) == "Data":
    BASE = os.path.dirname(BASE)

RAW_DIR = os.path.join(BASE, "Data", "Outputs", "Raw_Data")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Output directory: {RAW_DIR}")

Output directory: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL/Data/Outputs/Raw_Data


## Extract all tickers: t0 baseline (`nky_membership_2003-01-02.csv`) ∪ `nky_changes`

In [4]:
def normalize_ticker(bloomberg_ticker):
    """Extract ticker symbol from Bloomberg format.

    NKY rule:
      - 4-digit Tokyo codes are converted to yfinance form with .T suffix
        e.g. '1302 JT Equity' -> '1302.T'
    """
    if pd.isna(bloomberg_ticker) or not bloomberg_ticker:
        return ""
    token = str(bloomberg_ticker).strip().split()[0]
    if re.match(r"^\d{4}$", token):
        return f"{token}.T"
    return token


def extract_all_tickers_from_nky_log(filepath):
    """Parse nky_changes and extract all unique ticker symbols (ADD/DEL history only)."""
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"nky_changes not found: {filepath}")
    
    tickers = set()
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        in_add = False
        in_del = False
        
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Detect section headers
            if "ADD" in line.upper() and "DEL" not in line.upper():
                in_add = True
                in_del = False
                continue
            if "DEL" in line.upper() and "ADD" not in line.upper():
                in_del = True
                in_add = False
                continue
            
            # Skip metadata lines
            if "TOTAL" in line.upper() or "PERIOD" in line.upper():
                continue
            if re.match(r"^-+$", line):  # separator lines
                continue
            if "!!!" in line or "[END" in line:
                continue
            
            # Extract tickers from ADD/DEL sections
            if in_add or in_del:
                # Skip words that are not tickers
                skip_words = ("ADD", "DEL", "PERIOD", "TOTAL", "Equity", "COUNT", 
                              "File", "Edit", "Format", "View", "Help", "LEND")
                
                # Split by common separators
                parts = re.split(r"[+*•\-]", line)
                for part in parts:
                    part = part.strip()
                    if not part:
                        continue
                    
                    # Extract first token (ticker with possible suffix)
                    tokens = part.split()
                    if not tokens:
                        continue
                    
                    ticker_with_suffix = tokens[0]
                    
                    # Skip if it's a skip word
                    if ticker_with_suffix in skip_words:
                        continue
                    
                    # Normalize to get ticker symbol
                    ticker = normalize_ticker(ticker_with_suffix)
                    if ticker and re.match(r"^[A-Z0-9\.\-]{2,12}$", ticker.upper()):
                        tickers.add(ticker)
    
    return sorted(tickers)


# nky_changes (rebalances) + t0 baseline CSV (full Nikkei 225 at 2003-01-02)
DATA_DIR = os.path.join(BASE, "Data")
NKY_LOG = os.path.join(DATA_DIR, "nky_changes")
if not os.path.isfile(NKY_LOG):
    NKY_LOG = os.path.join(BASE, "nky_changes")

BASELINE_CSV = os.path.join(DATA_DIR, "nky_membership_2003-01-02.csv")

# Tickers from rebalance log
from_log = set(extract_all_tickers_from_nky_log(NKY_LOG))

# Tickers from t0 snapshot
df_b = pd.read_csv(BASELINE_CSV)
col = "ticker_bloomberg" if "ticker_bloomberg" in df_b.columns else df_b.columns[0]
from_baseline = {normalize_ticker(str(x).strip()) for x in df_b[col].astype(str)}
from_baseline = {t for t in from_baseline if t}

ALL_TICKERS = sorted(from_log | from_baseline)
print(f"\nUnique tickers: {len(ALL_TICKERS)} (from nky_changes: {len(from_log)}, t0 CSV: {len(from_baseline)})")
print(f"Sample: {ALL_TICKERS[:20]}")

# Identify Bloomberg-only IDs (numeric prefixes that won't work on yfinance)
bloomberg_only = [t for t in ALL_TICKERS if re.match(r"^\d{7,8}[A-Z]$", t)]
yfinance_compatible = [t for t in ALL_TICKERS if t not in bloomberg_only]

print(f"\nBloomberg-only IDs (will fail on yfinance): {len(bloomberg_only)}")
print(f"  {bloomberg_only}")
print(f"\nyfinance-compatible tickers: {len(yfinance_compatible)}")


Unique tickers: 318 (from nky_changes: 169, t0 CSV: 225)
Sample: ['1332.T', '1333.T', '1334.T', '1601.T', '1605.T', '1721.T', '1801.T', '1802.T', '1803.T', '1808.T', '1812.T', '1861.T', '1925.T', '1928.T', '1963.T', '2001.T', '2002.T', '2201.T', '2202.T', '2261.T']

Bloomberg-only IDs (will fail on yfinance): 1
  ['2552241D']

yfinance-compatible tickers: 317


## Download Data from yfinance

In [5]:
def download_ticker_data(ticker, start_date, end_date):
    """Download daily OHLCV data for a single ticker from yfinance.
    
    Returns:
        DataFrame with columns: date, open, high, low, close, volume
        None if download fails or data is empty
    """
    try:
        # Download data
        data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            interval="1d",
            progress=False
        )
        
        if data.empty:
            return None
        
        # Handle MultiIndex columns (yfinance returns MultiIndex even for single ticker)
        if isinstance(data.columns, pd.MultiIndex):
            # Flatten MultiIndex: ('Close', 'AAPL') -> 'Close'
            data.columns = data.columns.get_level_values(0)
        
        # Reset index to get date as a column
        data = data.reset_index()
        
        # Normalize column names to lowercase
        data.columns = [c.lower() for c in data.columns]
        
        # Select OHLCV columns (in correct order)
        required_cols = ['date', 'open', 'high', 'low', 'close', 'volume']
        available_cols = [c for c in required_cols if c in data.columns]
        
        if 'date' not in available_cols or 'close' not in available_cols:
            return None
        
        result = data[available_cols]
        
        # Ensure volume column exists (some tickers might not have it)
        if 'volume' not in result.columns:
            result['volume'] = 0
        
        return result
    
    except Exception as e:
        # Optionally print error for debugging
        # print(f"Error downloading {ticker}: {e}")
        return None

In [6]:
# Download data for all tickers
tickers_retrieved = []
tickers_not_retrieved = []
ticker_info = {}  # Track first/last date and row count

print(f"\nDownloading {len(ALL_TICKERS)} tickers...\n")

for ticker in tqdm(ALL_TICKERS, desc="Downloading"):
    df = download_ticker_data(ticker, START_DATE, END_DATE)
    
    if df is not None and len(df) > 0:
        # Save to CSV
        output_path = os.path.join(RAW_DIR, f"{ticker}.csv")
        df.to_csv(output_path, index=False)
        
        # Track info
        tickers_retrieved.append(ticker)
        ticker_info[ticker] = {
            'first_date': df['date'].min(),
            'last_date': df['date'].max(),
            'row_count': len(df)
        }
    else:
        tickers_not_retrieved.append(ticker)

print(f"\n{'='*60}")
print(f"DOWNLOAD SUMMARY")
print(f"{'='*60}")
print(f"Total tickers attempted: {len(ALL_TICKERS)}")
print(f"Successfully downloaded: {len(tickers_retrieved)}")
print(f"Failed/unavailable: {len(tickers_not_retrieved)}")
print(f"\nSuccess rate: {len(tickers_retrieved)/len(ALL_TICKERS)*100:.1f}%")

Downloading:   1%|          | 2/318 [00:01<02:51,  1.84it/s]$1334.T: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)

1 Failed download:
['1334.T']: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)
Downloading:   1%|          | 3/318 [00:01<01:54,  2.75it/s]$1601.T: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)

1 Failed download:
['1601.T']: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)
Downloading:   6%|▌         | 18/318 [00:09<02:50,  1.76it/s]$2202.T: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)

1 Failed download:
['2202.T']: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)
Downloading:   6%|▌         | 19/318 [00:09<02:17,  2.17it/s]$2261.T: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)

1 Failed download:
['2261.T']: possibly delisted; no price data found  (1d 2003-01-02 -> 2026-03-15)
Downloading:   9%|▉         | 28/3


DOWNLOAD SUMMARY
Total tickers attempted: 318
Successfully downloaded: 266
Failed/unavailable: 52

Success rate: 83.6%


In [7]:
# Show failed tickers breakdown
failed_bloomberg_ids = [t for t in tickers_not_retrieved if t in bloomberg_only]
failed_regular = [t for t in tickers_not_retrieved if t not in bloomberg_only]

print(f"\nFailed tickers breakdown:")
print(f"  Bloomberg IDs (expected to fail): {len(failed_bloomberg_ids)}")
print(f"    {failed_bloomberg_ids}")
print(f"\n  Regular tickers (delisted/unavailable): {len(failed_regular)}")
print(f"    {failed_regular}")


Failed tickers breakdown:
  Bloomberg IDs (expected to fail): 1
    ['2552241D']

  Regular tickers (delisted/unavailable): 51
    ['1334.T', '1601.T', '2202.T', '2261.T', '2536.T', '2779.T', '3102.T', '3404.T', '3893.T', '4010.T', '4501.T', '4505.T', '4511.T', '4795.T', '5001.T', '5002.T', '5405.T', '5407.T', '5413.T', '5701.T', '6502.T', '6764.T', '6767.T', '6773.T', '6796.T', '6933.T', '6991.T', '8003.T', '8028.T', '8183.T', '8232.T', '8238.T', '8264.T', '8270.T', '8307.T', '8332.T', '8355.T', '8403.T', '8404.T', '8583.T', '8603.T', '8606.T', '8752.T', '8755.T', '8815.T', '9062.T', '9205.T', '9437.T', '9613.T', '9681.T', '9737.T']


## Save Tracking Files

In [8]:
# Save tickers_retrieved.csv
df_retrieved = pd.DataFrame({
    'ticker': tickers_retrieved,
    'first_date': [ticker_info[t]['first_date'] for t in tickers_retrieved],
    'last_date': [ticker_info[t]['last_date'] for t in tickers_retrieved],
    'row_count': [ticker_info[t]['row_count'] for t in tickers_retrieved]
})
df_retrieved.to_csv(os.path.join(RAW_DIR, "tickers_retrieved.csv"), index=False)
print(f"Saved tickers_retrieved.csv ({len(tickers_retrieved)} tickers)")

# Save tickers_not_retrieved.csv
df_not_retrieved = pd.DataFrame({
    'ticker': tickers_not_retrieved,
    'reason': ['Bloomberg ID' if t in bloomberg_only else 'Unavailable/Delisted' 
               for t in tickers_not_retrieved]
})
df_not_retrieved.to_csv(os.path.join(RAW_DIR, "tickers_not_retrieved.csv"), index=False)
print(f"Saved tickers_not_retrieved.csv ({len(tickers_not_retrieved)} tickers)")

print(f"\nAll tracking files saved to: {RAW_DIR}")

Saved tickers_retrieved.csv (266 tickers)
Saved tickers_not_retrieved.csv (52 tickers)

All tracking files saved to: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL/Data/Outputs/Raw_Data


## Data Quality Check

In [9]:
# Show data coverage statistics
print("\nData Coverage Statistics:")
print(f"Earliest data: {df_retrieved['first_date'].min()}")
print(f"Latest data: {df_retrieved['last_date'].max()}")
print(f"\nRow count distribution:")
print(df_retrieved['row_count'].describe())

# Show tickers with limited data
limited_data = df_retrieved[df_retrieved['row_count'] < 1000].sort_values('row_count')
if len(limited_data) > 0:
    print(f"\nTickers with <1000 rows (recently listed or delisted):")
    print(limited_data[['ticker', 'first_date', 'last_date', 'row_count']].head(20))


Data Coverage Statistics:
Earliest data: 2003-01-02 00:00:00
Latest data: 2026-03-13 00:00:00

Row count distribution:
count     266.000000
mean     5442.150376
std      1011.536093
min        57.000000
25%      5756.000000
50%      5756.000000
75%      5756.000000
max      5756.000000
Name: row_count, dtype: float64

Tickers with <1000 rows (recently listed or delisted):
     ticker first_date  last_date  row_count
209  8303.T 2025-12-17 2026-03-13         57
225  8729.T 2025-09-29 2026-03-13        111
87   5016.T 2025-03-19 2026-03-13        241
140  6526.T 2022-10-12 2026-03-13        837
115  5831.T 2022-10-03 2026-03-13        843


## Sample Random Ticker

In [10]:
# Display a random ticker's data
if len(tickers_retrieved) > 0:
    random_ticker = tickers_retrieved[np.random.randint(0, len(tickers_retrieved))]
    df_sample = pd.read_csv(os.path.join(RAW_DIR, f"{random_ticker}.csv"))
    print(f"\nSample data for {random_ticker}:")
    print(f"Shape: {df_sample.shape}")
    print(f"\nFirst 5 rows:")
    print(df_sample.head())
    print(f"\nLast 5 rows:")
    print(df_sample.tail())
    print(f"\nColumn types:")
    print(df_sample.dtypes)


Sample data for 6981.T:
Shape: (5756, 6)

First 5 rows:
         date        open        high         low       close  volume
0  2003-01-02  359.773560  359.773560  359.773560  359.773560       0
1  2003-01-03  359.773560  359.773560  359.773560  359.773560       0
2  2003-01-06  363.552684  365.064342  358.261901  359.773560  378900
3  2003-01-07  369.599378  369.599378  355.994492  364.308594  258300
4  2003-01-08  360.529464  360.529464  351.459554  352.215363  203400

Last 5 rows:
            date    open    high     low   close    volume
5751  2026-03-09  3433.0  3511.0  3346.0  3473.0  10957300
5752  2026-03-10  3588.0  3672.0  3574.0  3672.0   8965500
5753  2026-03-11  3720.0  3861.0  3718.0  3758.0   6972000
5754  2026-03-12  3715.0  3734.0  3661.0  3720.0   5766600
5755  2026-03-13  3580.0  3661.0  3575.0  3650.0   6545200

Column types:
date          str
open      float64
high      float64
low       float64
close     float64
volume      int64
dtype: object
